<a href="https://colab.research.google.com/github/Maziger/Laksegate-master-thesis/blob/main/POC/sentiment_analysis/sentiment_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Energy News Sentiment Analysis Pipeline

A full pipeline that:
1. Collects energy-related news via RSS feeds
2. Scores sentiment with FinBERT
3. Downloads TTF natural gas price data
4. Aligns & correlates sentiment with price returns
5. Visualises results

In [ ]:
import os
from google.colab import userdata

user = "Maziger"
repo = "Laksegate-master-thesis"

# Remove local directory if it already exists (re-run safety)
if os.path.isdir(repo):
    !rm -rf {repo}

!git clone https://github.com/{user}/{repo}.git
%cd Laksegate-master-thesis/POC/

# Only install packages not pre-installed in Colab
!pip install -q feedparser yfinance

In [ ]:
import os
import re
import warnings
from datetime import datetime, timezone
from pathlib import Path

import feedparser
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from scipy import stats
from transformers import pipeline

warnings.filterwarnings('ignore')

# Output paths
DATA_DIR = Path('data')
OUTPUT_DIR = Path('output')
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print('Dependencies loaded.')

## Step 1 — News Collection

In [ ]:
ENERGY_KEYWORDS = [
    'electricity', 'natural gas', 'energy', 'power', 'renewable',
    'wind', 'solar', 'grid', 'TTF', 'EEX', 'Nord Pool',
    'carbon', 'CO2', 'EUA', 'LNG', 'crude', 'oil'
]

RSS_FEEDS = [
    # Primary (required by spec)
    'https://feeds.reuters.com/reuters/companyNews',
    # Energy-specialist sources
    'https://www.energymonitor.ai/feed/',
    'https://oilprice.com/rss/main',
    'https://www.eia.gov/rss/todayinenergy.xml',
    'https://www.rigzone.com/news/rss/rigzone_latest.aspx',
    # Broader business/environment sources — keyword-filtered below
    'https://www.carbonbrief.org/feed',
    'https://feeds.bbci.co.uk/news/business/rss.xml',
    'https://www.theguardian.com/environment/energy/rss',
]

def _entry_text(entry):
    """Combine title + summary for an RSS entry."""
    title = getattr(entry, 'title', '') or ''
    summary = getattr(entry, 'summary', '') or ''
    # Strip basic HTML tags
    summary = re.sub(r'<[^>]+>', ' ', summary)
    return title, summary, f'{title}. {summary}'

def _parse_date(entry):
    """Return a timezone-naive date string (YYYY-MM-DD) from an RSS entry."""
    for attr in ('published_parsed', 'updated_parsed'):
        t = getattr(entry, attr, None)
        if t:
            try:
                return datetime(*t[:6]).strftime('%Y-%m-%d')
            except Exception:
                pass
    return datetime.now().strftime('%Y-%m-%d')

def is_energy_related(text):
    text_lower = text.lower()
    return any(kw.lower() in text_lower for kw in ENERGY_KEYWORDS)

def fetch_news(feeds=RSS_FEEDS, min_articles=50):
    all_articles = []
    for url in feeds:
        print(f'Fetching: {url}')
        try:
            feed = feedparser.parse(url)
            for entry in feed.entries:
                title, summary, combined = _entry_text(entry)
                all_articles.append({
                    'date': _parse_date(entry),
                    'title': title.strip(),
                    'summary': summary.strip(),
                    'combined_text': combined.strip(),
                    'source_url': url,
                    'link': getattr(entry, 'link', ''),
                })
        except Exception as e:
            print(f'  Warning: could not fetch {url}: {e}')

    print(f'\nTotal articles fetched (all topics): {len(all_articles)}')

    # Filter to energy-related only
    energy_articles = [
        a for a in all_articles if is_energy_related(a['combined_text'])
    ]
    print(f'Energy-related articles:             {len(energy_articles)}')

    if len(energy_articles) < min_articles:
        print(f'\nWARNING: Only {len(energy_articles)} energy articles found '
              f'(target >= {min_articles}). Consider adding more RSS feeds.')

    return all_articles, energy_articles

all_raw, energy_articles = fetch_news()

df_raw = pd.DataFrame(all_raw)
df_raw.to_csv(DATA_DIR / 'raw_news.csv', index=False)
print(f'\nSaved {len(df_raw)} raw articles to data/raw_news.csv')
df_raw.head(3)

## Step 2 — Sentiment Scoring with FinBERT

In [ ]:
import torch

device = 0 if torch.cuda.is_available() else -1
print(f'Using {"GPU (cuda:0)" if device == 0 else "CPU"} for inference.')

print('Loading ProsusAI/finbert...')
finbert = pipeline(
    'text-classification',
    model='ProsusAI/finbert',
    top_k=None,       # return all three labels
    truncation=True,
    max_length=512,
    device=device,
)
print('Model loaded.')

def score_text(text):
    """Return (positive, negative, neutral, net_sentiment) for a text string."""
    if not text or not text.strip():
        return 0.0, 0.0, 1.0, 0.0
    results = finbert(text[:2000])  # pre-clip to avoid excessive tokenisation
    scores = {r['label']: r['score'] for r in results[0]}
    pos = scores.get('positive', 0.0)
    neg = scores.get('negative', 0.0)
    neu = scores.get('neutral',  0.0)
    return pos, neg, neu, pos - neg

def score_articles(articles):
    rows = []
    for i, art in enumerate(articles):
        if i % 10 == 0:
            print(f'  Scoring article {i+1}/{len(articles)}...')
        pos, neg, neu, net = score_text(art['combined_text'])
        rows.append({
            **art,
            'positive':      pos,
            'negative':      neg,
            'neutral':       neu,
            'net_sentiment': net,
        })
    return pd.DataFrame(rows)

print(f'Scoring {len(energy_articles)} energy articles...')
df_scored = score_articles(energy_articles)
df_scored.to_csv(DATA_DIR / 'scored_news.csv', index=False)
print(f'\nSaved scored results to data/scored_news.csv')
df_scored[['date', 'title', 'positive', 'negative', 'neutral', 'net_sentiment']].head(5)

## Step 3 — Price Data (TTF Natural Gas Futures)

In [ ]:
def fetch_price_data(ticker='TTF=F', period='90d'):
    print(f'Downloading {ticker} price data ({period})...')
    df = yf.download(ticker, period=period, auto_adjust=True, progress=False)

    if df.empty:
        raise ValueError(f'No price data returned for {ticker}. '
                         'The TTF=F ticker may not be available in your region — '
                         'try NG=F (Henry Hub) as an alternative.')

    # Flatten MultiIndex columns if present
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df[['Close']].copy()
    df.index = pd.to_datetime(df.index).normalize()   # date only
    df.index.name = 'date'
    df['return_pct'] = df['Close'].pct_change() * 100
    df = df.dropna(subset=['return_pct'])
    df.index = df.index.strftime('%Y-%m-%d')
    print(f'Price data: {len(df)} trading days, {df.index[0]} to {df.index[-1]}')
    return df

df_prices = fetch_price_data()
df_prices.to_csv(DATA_DIR / 'price_data.csv')
print('Saved to data/price_data.csv')
df_prices.tail(5)

## Step 4 — Alignment & Correlation

In [ ]:
# Aggregate sentiment to daily averages
df_daily_sentiment = (
    df_scored
    .groupby('date')
    .agg(
        avg_sentiment=('net_sentiment', 'mean'),
        avg_positive= ('positive',      'mean'),
        avg_negative= ('negative',      'mean'),
        avg_neutral=  ('neutral',       'mean'),
        article_count=('net_sentiment', 'count'),
    )
    .reset_index()
)
print(f'Sentiment aggregated over {len(df_daily_sentiment)} unique days')
df_daily_sentiment.head()

In [ ]:
# Reset price index to a plain column for merging
df_prices_reset = df_prices.reset_index()  # 'date' becomes a column

# Merge on date
df_merged = pd.merge(df_daily_sentiment, df_prices_reset, on='date', how='inner')

# Add next-day return (lag price return by -1)
df_merged = df_merged.sort_values('date').reset_index(drop=True)
df_merged['next_day_return'] = df_merged['return_pct'].shift(-1)

print(f'Merged rows (days with both sentiment and price data): {len(df_merged)}')
df_merged[['date', 'avg_sentiment', 'article_count', 'return_pct', 'next_day_return']].head(10)

In [ ]:
# Correlation analysis
df_corr_sameday = df_merged[['avg_sentiment', 'return_pct']].dropna()
df_corr_nextday = df_merged[['avg_sentiment', 'next_day_return']].dropna()

print('=== Correlation: avg_sentiment vs same-day return ===')
corr_matrix_sameday = df_corr_sameday.corr()
print(corr_matrix_sameday.to_string())

print('\n=== Correlation: avg_sentiment vs next-day return (1-day lag) ===')
corr_matrix_nextday = df_corr_nextday.rename(
    columns={'next_day_return': 'next_day_return (t+1)'}
).corr()
print(corr_matrix_nextday.to_string())

# Extract scalar correlations for the summary
r_sameday = df_corr_sameday.corr().loc['avg_sentiment', 'return_pct']
r_nextday  = df_corr_nextday.corr().loc['avg_sentiment', 'next_day_return']

## Step 5 — Visualisation

In [ ]:
# Plot 1: Daily sentiment bar chart + price return line (dual y-axis)
df_plot = df_merged.dropna(subset=['avg_sentiment', 'return_pct']).copy()
df_plot['date_dt'] = pd.to_datetime(df_plot['date'])

fig, ax1 = plt.subplots(figsize=(14, 5))

colors = ['#d73027' if v < 0 else '#1a9850' for v in df_plot['avg_sentiment']]
ax1.bar(df_plot['date_dt'], df_plot['avg_sentiment'],
        color=colors, alpha=0.75, label='Daily avg net sentiment', zorder=2)
ax1.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax1.set_xlabel('Date')
ax1.set_ylabel('Avg Net Sentiment (positive − negative)', color='#1a9850')
ax1.tick_params(axis='y', labelcolor='#1a9850')

ax2 = ax1.twinx()
ax2.plot(df_plot['date_dt'], df_plot['return_pct'],
         color='#4575b4', linewidth=1.5, marker='o', markersize=3,
         label='TTF daily return (%)', zorder=3)
ax2.set_ylabel('TTF Return (%)', color='#4575b4')
ax2.tick_params(axis='y', labelcolor='#4575b4')

ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax1.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
fig.autofmt_xdate(rotation=45)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)

plt.title('Daily Energy News Sentiment vs TTF Natural Gas Return')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sentiment_vs_price.png', dpi=150)
plt.show()
print('Saved: output/sentiment_vs_price.png')

In [ ]:
# Plot 2: Scatter — avg_sentiment vs next-day return with regression line
df_scatter = df_merged.dropna(subset=['avg_sentiment', 'next_day_return']).copy()

x = df_scatter['avg_sentiment'].values
y = df_scatter['next_day_return'].values

slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
x_line = np.linspace(x.min(), x.max(), 200)
y_line = slope * x_line + intercept

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(x, y, alpha=0.65, edgecolors='white', linewidth=0.4,
           color='#4575b4', s=60, label='Observations', zorder=3)
ax.plot(x_line, y_line, color='#d73027', linewidth=2,
        label=f'Regression (r={r_value:.3f}, p={p_value:.3f})')
ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
ax.axvline(0, color='grey', linestyle='--', linewidth=0.8)
ax.set_xlabel('Daily Avg Net Sentiment (positive − negative)')
ax.set_ylabel('Next-Day TTF Return (%)')
ax.set_title('Energy Sentiment vs Next-Day TTF Natural Gas Return')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sentiment_vs_nextday_return.png', dpi=150)
plt.show()
print('Saved: output/sentiment_vs_nextday_return.png')

## Summary

In [ ]:
date_range_start = df_merged['date'].min()
date_range_end   = df_merged['date'].max()

print('=' * 55)
print('ENERGY NEWS SENTIMENT PIPELINE — SUMMARY')
print('=' * 55)
print(f'Articles fetched (all topics)  : {len(all_raw)}')
print(f'Articles passing energy filter : {len(energy_articles)}')
print(f'Date range covered             : {date_range_start} to {date_range_end}')
print(f'Days with merged data          : {len(df_merged)}')
print(f'Correlation (sentiment / same-day return)   : {r_sameday:+.4f}')
print(f'Correlation (sentiment / next-day return)   : {r_nextday:+.4f}')
print('=' * 55)

if abs(r_nextday) > abs(r_sameday):
    print('=> Sentiment has STRONGER predictive correlation with next-day returns.')
else:
    print('=> Sentiment has STRONGER (or equal) correlation with same-day returns.')